# Setup Cell
Run this first to load the dataset and prepare `X_train` and `y_train` as 1D arrays for binary classification.

In [1]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Ensure we are in the project root to import src natively
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
sys.path.append(os.getcwd())

from src.preprocess import clean_text

print("Loading dataset...")
data_path = r"..\data\archive (6)\train.csv"
df_full = pd.read_csv(data_path)

print("Sampling 20,000 rows (stratified)...")
df, _ = train_test_split(
    df_full,
    train_size=20000,
    stratify=df_full["toxic"],
    random_state=42
)

print("Cleaning text...")
df["comment_text"] = df["comment_text"].astype(str).apply(clean_text)

print("Splitting train/test...")

X_train, X_test, y_train, y_test = train_test_split(
    df["comment_text"].tolist(),
    df["toxic"].values,  # <--- ONLY the single "toxic" column (Binary)
    test_size=0.2,
    random_state=42,
    stratify=df["toxic"]
)

print(f"X_train samples: {len(X_train)}")
print(f"y_train shape:   {y_train.shape}")
print("Setup complete! Proceed to the next cell.")

Loading dataset...
Sampling 20,000 rows (stratified)...
Cleaning text...
Splitting train/test...
X_train samples: 16000
y_train shape:   (16000,)
Setup complete! Proceed to the next cell.


# Model Training & Tuning Cell
This cell trains a standard, binary `LogisticRegression` classifer with GridSearchCV to tune hyperparameters. It drops `OneVsRestClassifier` since we are not doing multi-label classification.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import pandas as pd
import json

# Build pipeline (Removed OneVsRestClassifier)
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=3
    )),
    ("clf", LogisticRegression(
        class_weight="balanced",   # handles 90/10 imbalance natively
        max_iter=1000,
        random_state=42
    ))
])

# Hyperparameter grid (Renamed clf__estimator__C to clf__C)
param_grid = {
    "tfidf__max_features": [30000, 50000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.1, 1.0, 10.0],
}

# Run grid search with 3-fold cross-validation
print("Running GridSearchCV — this takes ~2-5 minutes...")
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\n✅ Best parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")
print(f"   Best CV F1 (macro): {grid_search.best_score_:.4f}")

# Evaluate best model on held-out test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# Compute binary metrics
acc = best_model.score(X_test, y_test)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")
roc_auc = roc_auc_score(y_test, y_proba)

print("\n" + "="*60)
print("BINARY BASELINE RESULTS — TF-IDF + Logistic Regression")
print("="*60)
print(f"  Accuracy:    {acc:.4f}")
print(f"  Macro F1:    {f1_macro:.4f}")
print(f"  Weighted F1: {f1_weighted:.4f}")
print(f"  ROC-AUC:     {roc_auc:.4f}")
print("="*60)

# Save metrics to outputs/
metrics = {
    "model": "TF-IDF + Logistic Regression (Binary Tuned)",
    "best_params": grid_search.best_params_,
    "metrics": {
       "accuracy": round(acc, 4),
       "f1_macro": round(f1_macro, 4),
       "f1_weighted": round(f1_weighted, 4),
       "roc_auc_score": round(roc_auc, 4)
    }
}

import os
os.makedirs("outputs", exist_ok=True)
with open("outputs/baseline_metrics_binary.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\n✅ Metrics saved to outputs/baseline_metrics_binary.json")
print("📋 Ready to copy these exactly into paper.md!")

Running GridSearchCV — this takes ~2-5 minutes...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

✅ Best parameters found:
   clf__C: 10.0
   tfidf__max_features: 30000
   tfidf__ngram_range: (1, 1)
   Best CV F1 (macro): 0.8487

BINARY BASELINE RESULTS — TF-IDF + Logistic Regression
  Accuracy:    0.9475
  Macro F1:    0.8477
  Weighted F1: 0.9474
  ROC-AUC:     0.9514

✅ Metrics saved to outputs/baseline_metrics_binary.json
📋 Ready to copy these exactly into paper.md!
